# Luma using DTL workflow demonstration for Kalbar area

In [ ]:
!python -m pip install .. --quiet

In [ ]:
import ee 

ee.Authenticate() 
ee.Initialize()

# 1. Satellite imagery

In [ ]:
import geemap
from luma_ge.data_acquisition import Reflectance_Data, final_Image

# AOI definition

aoi = geemap.shp_to_ee('../data/modular_mapping_approach/pagaralam_test/Kota_Pagar_Alam.shp')

#========== FIRST RETRIVE THE MULTISPECTRAL BAND===========
#Intialize the relfectance class data function
optical_reflectance = Reflectance_Data()
#Initialize the final image class for composite creation
composite = final_Image() #NEW FEATURE ADDED HERE
#define the start and end date for imagery collection
start = '2024-01-01'
end = '2024-12-31'
#get the image collection and corresponding statistics
landsat_data, meta = optical_reflectance.get_optical_data(aoi, start, end, optical_data='L8_SR', 
                                                           cloud_cover=40, compute_detailed_stats=False)
#create mosaic between image collection, and clip based on AOI
mosaic_landsat = composite.get_quality_mosaic(landsat_data, aoi, quality_band= 'NDVI', calculate_coverage=False) #REPLACE OLD CODE WITH THE NEW ONE HERE
#Alternatively you can use temporal aggregation (ee reducer) to create mode cloudless imagery
#Add new functionality to calculate the coverage of the composite
median_landsat, coverage = composite.get_temporal_composite(landsat_data, aoi, reducer='Median', calculate_coverage=True) #REPLACE OLD CODE WITH THE NEW ONE HERE
#visualization parameter
l8_sr_visparam = {'min': 0,'max': 0.4,'gamma': [0.95, 1.1, 1],'bands':['NIR', 'RED', 'GREEN']}

#retive thermal bands from TOA
thermal_bands, thermal_stats = optical_reflectance.get_thermal_bands(aoi, start, end, cloud_cover=40, thermal_data='L8_TOA', compute_detailed_stats=False)
median_thermal = composite.get_temporal_composite(thermal_bands, aoi, reducer='Median') #REPLACE THE OLD CODE WITH THE NEW ONE
thermal_vis = {'min': 286,'max': 300,'gammma': 0.4}
#stacked all landsat bands and convert to float(making sure all data type are compatible)
stacked_landsat = median_landsat.addBands(median_thermal).toFloat()


# 2. Classification scheme 

In [ ]:
from luma_ge.classification_scheme import LULC_Scheme_Manager

manager = LULC_Scheme_Manager()

scheme_name = "Epistem"
success, message = manager.load_default_scheme(scheme_name)
classification_df = manager.get_dataframe()

print(classification_df.to_string(index=False))

# 3. Upload modular reference data

In [ ]:
import pandas as pd
import numpy as np

df_train_csv = '../data/modular_mapping_approach/pagaralam_test/pagaralam_test_revengineer.csv'
df_train = pd.read_csv(df_train_csv)

## Define default scheme labelling ruleset

In [ ]:
# ── DEFINE RULESET ─────────────────────────────────────────────────────────
# Each row = one class rule. Columns are primitives with operators.
# Format: ">0.40" means "greater than 0.40"
#         "==1" means "equal to 1"
#         ">=25" means "greater than or equal to 25"
#         'treecover': '<30 | >80',  means treecover < 30 OR treecover > 80   
#         None means "no condition on this primitive"

ruleset = pd.DataFrame([
    {
        'class_id': 23,
        'class_name': 'Waterbody',
        'priority': 1,
        'waterbody_presence': '>40',
        # 'waterbody_artificiality': '==0'
    },
    {
        'class_id': 20,
        'class_name': 'Fish Pond',
        'priority': 2,
        # 'waterbody_artificiality': '==1',
        'waterbody_presence': '>40',
    },
    {
        'class_id': 20,
        'class_name': 'Settlement',
        'priority': 3,
        'waterbody_presence': '<40',
        'builtup_presence': '>0.9'
    },
    {
        'class_id': 21,
        'class_name': 'Cleared Land',
        'priority': 4,
        'waterbody_presence': '<40',
        'bareSoil_presence': '>0.9',
        'mining': '0'
    },
    {
        'class_id': 22,
        'class_name': 'Mining area',
        'priority': 5,
        'waterbody_presence': '<40',
        'bareSoil_presence': '>0.9',
        'mining': '1'
    },
    {
        'class_id': 1,
        'class_name': 'Primary Dryland Forest',
        'priority': 6,
        'waterbody_presence': '<10',
        'tree_cover': '>0.9',
        'agricultural_activity_type': '==0',
    },
    {
        'class_id': 3,
        'class_name': 'Primary Mangrove Forest',
        'priority': 7,
        'waterbody_presence': '>10',
        'tree_cover': '>0.9',
        'mangrove_presence': '==1',
        'agricultural_activity_type': '==0',
    },
    {
        'class_id': 5,
        'class_name': 'Primary Swamp Forest',
        'priority': 8,
        'waterbody_presence': '>10',
        'tree_cover': '>0.9',
        'agricultural_activity_type': '==0',
    },
    {
        'class_id': 2,
        'class_name': 'Secondary Dryland Forest',
        'priority': 9,
        'waterbody_presence': '<10',
        'tree_cover': '>0.2',
        'agricultural_activity_type': '==0',        
    },
    {
        'class_id': 4,
        'class_name': 'Secondary Mangrove Forest',
        'priority': 10,
        'waterbody_presence': '>10',
        'tree_cover': '>0.2',
        'mangrove_presence': '==1',
        'agricultural_activity_type': '==0',          
    },
    {
        'class_id': 6,
        'class_name': 'Secondary Swamp Forest',
        'priority': 11,
        'waterbody_presence': '>10',
        'tree_cover': '>0.2',
        'agricultural_activity_type': '==0',  
    },
    {
        'class_id': 8,
        'class_name': 'Rubber monoculture',
        'priority': 12,
        'waterbody_presence': '<40',
        'tree_cover': '>0.2',
        'rubber_presence': '==1',
        'agricultural_activity_type': '==1',          
    },
    {
        'class_id': 15,
        'class_name': 'Rubber agroforestry',
        'priority': 13,
        'waterbody_presence': '<40',
        'tree_cover': '>0.2',
        'rubber_presence': '==1',
        'other_species_presence': '==1',
        'agricultural_activity_type': '==1',  
    },
    {
        'class_id': 14,
        'class_name': 'Coffee agroforestry',
        'priority': 14,
        'waterbody_presence': '<40',
        'coffee_presence': '==1',
        'agricultural_activity_type': '==1',  
    },
    {
        'class_id': 10,
        'class_name': 'Cacao monoculture',
        'priority': 15,
        'waterbody_presence': '<40',
        'tree_cover': '>0.2',
        'cacao_presence': '==1',
        'agricultural_activity_type': '==1',  
    },
    {
        'class_id': 12,
        'class_name': 'Other monoculture',
        'priority': 16,
        'waterbody_presence': '<40',
        'tree_cover': '>0.2',
        'other_species_presence': '==1',
        'agricultural_activity_type': '==1',  
    },
    {
        'class_id': 9,
        'class_name': 'Oil palm monoculture',
        'priority': 17,
        'waterbody_presence': '<40',
        # 'tree_cover': '>0.2',
        'oilpalm_presence': '==1',
        'agricultural_activity_type': '==1',
        'tree_horizonal_pattern': '1',
    },
    {
        'class_id': 11,
        'class_name': 'Coconut monoculture',
        'priority': 18,
        'waterbody_presence': '<40',
        # 'tree_cover': '>0.2',
        'coconut_presence': '==1',
        'agricultural_activity_type': '==1',
        'tree_horizonal_pattern': '2',
    },
    {
        'class_id': 7,
        'class_name': 'Plantation forest',
        'priority': 19,
        'waterbody_presence': '<40',
        'tree_cover': '>0.2',
        # 'timber_extraction_rotation': '==1', # DOUBLE CHECK
        'agricultural_activity_type': '==0',  
    },
    {
        'class_id': 19,
        'class_name': 'Shrub',
        'priority': 20,
        'waterbody_presence': '<40',
        'tree_cover': '<0.2',
        'shrub_presence': '>0.2',
        'agricultural_activity_type': '==0',  
    },
    {
        'class_id': 18,
        'class_name': 'Grass or Savanna',
        'priority': 21,
        'waterbody_presence': '<40',
        'tree_cover': '<0.2',
        'herb_presence': '>0.2',
        'agricultural_activity_type': '==0',  
    },
    {
        'class_id': 13,
        'class_name': 'Other Cropland',
        'priority': 22,
        'waterbody_presence': '<40',
        'tree_cover': '<0.2',
        'herb_presence': '>0.2',
        'agricultural_activity_type': '==1',  
    },
    {
        'class_id': 17,
        'class_name': 'Paddy field',
        'priority': 23,
        # 'waterbody_presence': '<40',
        'tree_cover': '<0.2',
        'graminoid_presence': '>0.2',
        'agricultural_activity_type': '==1', 
    },
    {
        'class_id': 16,
        'class_name': 'Mixed/home garden',
        'priority': 24,
        'agricultural_activity_type': '==1', 
    }   
])

print('✓ Ruleset defined')
print(f'  Total rules: {len(ruleset)}')
print(f'  Priority order: {list(ruleset["class_name"])}\n')

ruleset = ruleset.sort_values(by='priority', ascending=True)
ruleset

## Helper functions to label the classes

In [ ]:
# ── HELPER FUNCTIONS ───────────────────────────────────────────────────────

def safe_num(val, default=0):
    """Convert value to float, return default if None or NaN."""
    if val is None:
        return default
    if isinstance(val, (int, float)):
        if np.isnan(val):
            return default
        return float(val)
    try:
        return float(val)
    except:
        return default

def evaluate_condition(row_val, condition_str):
    """
    Evaluate a single condition: row_val op threshold?
    Supports OR logic with pipe separator: ">0.40|<0.10"
    
    Args:
        row_val: The value from the sample
        condition_str: String like ">0.40", ">=25", ">0.40|<0.10"
    
    Returns:
        bool: True if condition is satisfied, False otherwise
    """
    if condition_str is None:
        return True  # No condition → always passes
    
    condition_str = str(condition_str).strip()
    
    # Handle OR logic (pipe-separated conditions)
    if '|' in condition_str:
        sub_conditions = [c.strip() for c in condition_str.split('|')]
        return any(evaluate_condition(row_val, c) for c in sub_conditions)
    
    # Parse operator and threshold
    if condition_str.startswith('=='):
        op, threshold_str = '==', condition_str[2:]
    elif condition_str.startswith('>='):
        op, threshold_str = '>=', condition_str[2:]
    elif condition_str.startswith('<='):
        op, threshold_str = '<=', condition_str[2:]
    elif condition_str.startswith('>'):
        op, threshold_str = '>', condition_str[1:]
    elif condition_str.startswith('<'):
        op, threshold_str = '<', condition_str[1:]
    else:
        return True  # Invalid condition → pass
    
    threshold = safe_num(threshold_str)
    row_val_num = safe_num(row_val)
    
    if op == '>':
        return row_val_num > threshold
    elif op == '>=':
        return row_val_num >= threshold
    elif op == '<':
        return row_val_num < threshold
    elif op == '<=':
        return row_val_num <= threshold
    elif op == '==':
        return row_val_num == threshold
    
    return False

def check_rule_match(row, rule):
    """
    Check if a sample row matches all conditions in a rule.
    
    Args:
        row: pd.Series with sample data
        rule: pd.Series with rule conditions
    
    Returns:
        bool: True if ALL conditions are satisfied
    """
    # Get all primitive columns (skip metadata like class_id, class_name, priority)
    metadata = {'class_id', 'class_name', 'priority'}
    primitive_cols = [col for col in rule.index if col not in metadata and col != 'treecover_max']
    
    for prim in primitive_cols:
        condition = rule[prim]
        
        # Handle special case for range checks (e.g., treecover_max)
        if pd.isna(condition) or condition is None:
            continue  # No condition on this primitive
        
        row_val = row.get(prim, np.nan)
        
        if not evaluate_condition(row_val, condition):
            return False  # Any condition fails → rule doesn't match
    
    return True  # All conditions passed

print('✓ Helper functions defined')

## Assign the class labels to the modular reference data

In [ ]:
# ── ASSIGN LABELS VIA DECISION RULES ───────────────────────────────────────

def assign_label(row, ruleset):
    """
    Evaluate all rules in priority order. First match → assign that class.
    If no match → return 0 (unclassified).
    
    This mirrors GEE's nested ee.Algorithms.If logic:
    Each rule's NO branch continues to the next rule.
    First YES match stops evaluation.
    """
    # Sort by priority to ensure correct evaluation order
    ruleset_sorted = ruleset.sort_values('priority').reset_index(drop=True)
    
    for _, rule in ruleset_sorted.iterrows():
        if check_rule_match(row, rule):
            return rule['class_id']
    
    return 0  # No rule matched → unclassified

# Apply to all training samples
print('[Step 3] Assigning labels via decision rules...')

# # IF only use a subset of a class, then apply this instead:
# ruleset_subset = ruleset[ruleset['class_id'].isin([23, 20, 13])]  # Only 3 classes

df_train['label'] = df_train.apply(lambda row: assign_label(row, ruleset), axis=1)
# ruleset_subset = ruleset[ruleset['class_id'].isin([2, 4, 9, 13, 18, 19, 20, 21, 23])]
# df_split_train['label'] = df_split_train.apply(lambda row: assign_label(row, ruleset_subset), axis=1)

print(f'✓ Labels assigned to {len(df_train)} samples\n')
print('Label distribution (assigned):')
print(df_train['label'].value_counts().sort_index())
print(f'\nUnclassified (label=0): {(df_train["label"] == 0).sum()}')